# 05 — Explainability and Retention Insights

This notebook generates global feature importance and four complementary visualizations when supported by the installed SHAP version. Interpretations describe model behavior and association; they do not establish causality. Run after the model and processed splits exist.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
ROOT=Path.cwd(); REPORTS=ROOT/"reports"; FIG=REPORTS/"figures"; FIG.mkdir(parents=True,exist_ok=True)
pack=joblib.load(ROOT/"models"/"churn_model.joblib"); model=pack["model"]
X=pd.read_csv(ROOT/"data"/"processed"/"test.csv").drop(columns=["churn"],errors="ignore")
print("Model:",pack.get("model_name"),"| threshold:",pack.get("threshold"),"| test sample:",X.shape)

## Global permutation importance

Permutation importance is model-agnostic and measures how much a chosen score changes when one feature is shuffled. This may be expensive on larger datasets; use a representative sample.

In [ ]:
from sklearn.inspection import permutation_importance
y=pd.read_csv(ROOT/"data"/"processed"/"test.csv")["churn"].astype(int)
sample=X.sample(min(600,len(X)),random_state=42); ys=y.loc[sample.index]
perm=permutation_importance(model,sample,ys,n_repeats=3,random_state=42,scoring="average_precision",n_jobs=-1)
imp=pd.DataFrame({"feature":sample.columns,"importance_mean":perm.importances_mean,"importance_std":perm.importances_std}).sort_values("importance_mean",ascending=False)
imp.to_csv(REPORTS/"permutation_importance.csv",index=False)
fig,ax=plt.subplots(figsize=(8,5)); top=imp.head(15).sort_values("importance_mean"); ax.barh(top.feature,top.importance_mean,xerr=top.importance_std); ax.set_title("1. Permutation importance (average precision)"); ax.set_xlabel("Decrease in score after permutation"); plt.tight_layout(); plt.savefig(FIG/"shap_visual_1_permutation_importance.png",dpi=160); plt.show()

## SHAP explanations

Because the saved pipeline includes preprocessing, SHAP support varies by estimator and library version. The following block tries the model-agnostic KernelExplainer on a small sample, which can be slow. If it fails, the permutation importance chart above remains available and the error should be recorded rather than hidden.

In [ ]:
try:
    import shap
    # Explain probability of churn (class 1) for a small, representative sample.
    background=X.sample(min(40,len(X)),random_state=7)
    explain_rows=X.sample(min(80,len(X)),random_state=17)
    explainer=shap.Explainer(lambda arr: model.predict_proba(pd.DataFrame(arr,columns=X.columns))[:,1], background, algorithm="permutation")
    sv=explainer(explain_rows, max_evals=max(2*X.shape[1]+1, 100))
    # Visual 2: SHAP summary/beeswarm
    plt.figure(); shap.plots.beeswarm(sv,max_display=15,show=False); plt.tight_layout(); plt.savefig(FIG/"shap_visual_2_beeswarm.png",dpi=160,bbox_inches="tight"); plt.show()
    # Visual 3: mean absolute SHAP values
    plt.figure(); shap.plots.bar(sv,max_display=15,show=False); plt.tight_layout(); plt.savefig(FIG/"shap_visual_3_global_bar.png",dpi=160,bbox_inches="tight"); plt.show()
    # Visual 4: dependence plot for top feature
    vals=np.abs(np.asarray(sv.values)).mean(axis=0); top_idx=int(np.argmax(vals)); top_feature=X.columns[top_idx]
    shap.plots.scatter(sv[:,top_feature], color=sv, show=False); plt.tight_layout(); plt.savefig(FIG/"shap_visual_4_dependence.png",dpi=160,bbox_inches="tight"); plt.show()
    # Visual 5: one local explanation/waterfall
    shap.plots.waterfall(sv[0],max_display=12,show=False); plt.tight_layout(); plt.savefig(FIG/"shap_visual_5_local_waterfall.png",dpi=160,bbox_inches="tight"); plt.show()
except Exception as e:
    print("SHAP visuals could not be generated in this environment:",repr(e))
    print("Install compatible shap/numpy versions and rerun. Do not claim missing plots were generated.")

## Retention strategy hypotheses
1. Offer onboarding help to high-risk customers with short tenure and month-to-month contracts.
2. Offer a transparent plan-fit and charge review where monthly charges are high relative to customer needs.
3. Offer relevant support/security setup assistance when service configuration suggests an unmet need.

Each strategy should be tested with a controlled experiment and evaluated using retention, customer satisfaction, contact cost and margin. See `docs/RETENTION_STRATEGIES.md` for operational safeguards.